## Power BI Export
- Purpose: Consolidate all EDA outputs into 4 clean tables for Power BI
- Input: EDA 01, 02, 03 processed CSV files
- Output: 4 tables saved to `../data/processed/powerbi/`

## Table Structure
| Table | Source | Description |
|-------|--------|-------------|
| `dim_date` | EDA01 | Date, CPI, CPI_MoM, Cancel Search |
| `fact_churn` | EDA01+02+03 | Churn rates by segment, price tier, type |
| `fact_substitution` | EDA03 | Netflix vs Ad-free search trends |
| `fact_simulation` | EDA03 | Price increase scenarios + revenue index |

In [1]:
import pandas as pd
import numpy as np
import os

# Output directory 
output_dir = '../data/processed/powerbi/'
os.makedirs(output_dir, exist_ok=True)
print(f'Output directory ready: {output_dir}')

Output directory ready: ../data/processed/powerbi/


### Table 1: dim_date
- Date dimension table
- Contains: CPI trend + Cancel search intent
- Used in: Page 1 Lag Chart

In [2]:
# Load CPI data
cpi = pd.read_csv('../data/references/CPIRECSL_consumerPriceIndex_2023_2026.csv')
cpi.columns = ['date', 'CPI']
cpi['date'] = pd.to_datetime(cpi['date'])

# Filter from 2024 onwards (align with search data)
cpi = cpi[cpi['date'] >= '2024-01-01'].copy()
cpi['CPI_MoM'] = cpi['CPI'].pct_change(fill_method=None) * 100

# Load cancel search data
search = pd.read_csv('../data/references/time_series_US_20240101-20260222-1958.csv')
search.columns = ['date', 'cancel_search']
search['date'] = pd.to_datetime(search['date'])

# Merge on date
dim_date = pd.merge(cpi, search, on='date', how='outer').sort_values('date')
dim_date = dim_date.round(4)

# Save
dim_date.to_csv(f'{output_dir}dim_date.csv', index=False)
print(f'dim_date: {dim_date.shape}')
print(dim_date.head(3))

dim_date: (26, 4)
        date      CPI  CPI_MoM  cancel_search
0 2024-01-01  137.798      NaN             35
1 2024-02-01  138.000   0.1466             37
2 2024-03-01  137.868  -0.0957             28


### Table 2: fact_churn
- Churn analysis unified table
- Combines: Segment churn (EDA01) + Family/Individual (EDA02) + Segment x Price Tier (EDA03)
- Used in: Page 1 Bar Chart, Page 3 Threshold

In [3]:
# Source 1: Segment churn (EDA01) 
seg_churn = pd.read_csv('../data/processed/powerbi/eda01_segment_churn.csv')
seg_churn.columns = ['Segment', 'Churn_Rate']
seg_churn['Segment_Type'] = 'All'
seg_churn['PriceTier'] = 'All'
seg_churn['Source'] = 'EDA01_Segment'

# Source 2: Family vs Individual (EDA02) 
seg_type = pd.read_csv('../data/processed/powerbi/eda_02_churn_survive_rate.csv')
seg_type['Segment'] = 'All'
seg_type['PriceTier'] = 'All'
seg_type['Source'] = 'EDA02_SegmentType'
seg_type = seg_type[['Segment', 'PriceTier', 'Segment_Type', 'Churn_Rate', 'Source']]

# Source 3: Segment x Price Tier (EDA03) 
seg_tier = pd.read_csv('../data/processed/powerbi/eda_03_churn_by_segment_tier.csv')
seg_tier['Source'] = 'EDA03_SegmentTier'
seg_tier = seg_tier[['Segment', 'PriceTier', 'Segment_Type', 'Churn_Rate', 'Source']]

# Combine all 
fact_churn = pd.concat([seg_churn[['Segment','PriceTier','Segment_Type','Churn_Rate','Source']], 
                        seg_type, 
                        seg_tier], ignore_index=True)
fact_churn['Churn_Rate'] = fact_churn['Churn_Rate'].round(4)

# Save
fact_churn.to_csv(f'{output_dir}fact_churn.csv', index=False)
print(f'fact_churn: {fact_churn.shape}')
print(fact_churn['Source'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/powerbi/eda01_segment_churn.csv'

### Table 3: fact_substitution
- Substitution threat timeseries
- Contains: Netflix alternative vs Ad-free YouTube search
- Used in: Page 1 Substitution Chart

In [ ]:
# Load substitution data
fact_substitution = pd.read_csv('../data/processed/powerbi/eda_03_substitution.csv')
fact_substitution.columns = ['date', 'netflix_alternative', 'adfree_youtube']
fact_substitution['date'] = pd.to_datetime(fact_substitution['date'])

# Add policy flag
fact_substitution['post_policy'] = fact_substitution['date'] >= '2025-10-01'

# Save
fact_substitution.to_csv(f'{output_dir}fact_substitution.csv', index=False)
print(f'fact_substitution: {fact_substitution.shape}')
print(fact_substitution.tail(3))

fact_substitution: (24, 4)
         date  netflix_alternative  adfree_youtube  post_policy
21 2025-12-01                   48              68         True
22 2026-01-01                   50              59         True
23 2026-02-01                   35              47         True


### Table 4: fact_simulation
- Price increase scenario simulation
- Contains: Revenue index + Churn by scenario + Price threshold
- Used in: Page 2 Simulator + Page 2 Threshold Chart

In [ ]:
# Load simulation results
simulation = pd.read_csv('../data/processed/powerbi/eda_03_simulation_results.csv')

# Load price threshold
threshold = pd.read_csv('../data/processed/powerbi/eda_03_price_threshold.csv')
threshold.columns = ['PriceTierFine', 'Threshold_Churn_Rate', 'CI_Low', 'CI_High']

# Save separately (different shapes, can't merge)
simulation.to_csv(f'{output_dir}fact_simulation.csv', index=False)
threshold.to_csv(f'{output_dir}fact_threshold.csv', index=False)

print(f'fact_simulation: {simulation.shape}')
print(f'fact_threshold: {threshold.shape}')
print(simulation)

fact_simulation: (4, 8)
fact_threshold: (5, 4)
  Price Increase  Family Price  Ind Price  Family Churn  Ind Churn  \
0            +5%         24.14      14.69         16.75      25.58   
1           +10%         25.29      15.39         16.84      25.84   
2           +17%         26.90      16.37         17.19      26.71   
3           +28%         29.43      17.91         17.88      28.45   

   Family Rev Index  Ind Risk Index  Total Rev Index  
0           11.9329          1.5267          16.3734  
1           12.4869          1.6154          17.1228  
2           13.2265          1.7762          18.0995  
3           14.3485          2.0698          19.5530  


### Export Summary

In [ ]:
print('Power BI Export Complete!')
tables = {
    'dim_date':          'date + CPI + search cancel  →  Page 1 Lag Chart',
    'fact_churn':        'churn integration           →  Page 1 Bar + Page 3',
    'fact_substitution': 'substitution threat         →  Page 1 Substitution',
    'fact_simulation':   'scenario simulation         →  Page 2 Simulator',
    'fact_threshold':    'price threshold             →  Page 2 Threshold'
}
for table, desc in tables.items():
    print(f'  ✅  {table:<22}  {desc}')

Power BI Export Complete!
  ✅  dim_date                date + CPI + search cancel  →  Page 1 Lag Chart
  ✅  fact_churn              churn integration           →  Page 1 Bar + Page 3
  ✅  fact_substitution       substitution threat         →  Page 1 Substitution
  ✅  fact_simulation         scenario simulation         →  Page 2 Simulator
  ✅  fact_threshold          price threshold             →  Page 2 Threshold
